### The main goal of "Multi Query Translation" is to take the input question and to translate it in some way, as to improve retrieval

## Setup and Indexing (Common to all)

In [69]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

In [70]:
# Load blog

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs = loader.load()

In [71]:
# Split 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

In [72]:
# Index 

vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

## 1) Multi-Query

![Multi-Query Image](Images/Multi-Query.png)

#### Multi-Query generates multiple alternative versions of a user question to improve document retrieval.

### Query Generation

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [6]:
# Multi Query: Different Perspectives 

#Prompt to generate 5 variations 

template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}""" 

prompt_perspectives = ChatPromptTemplate.from_template(template)

In [7]:
# Chain to generate the queries and split them into a Python list 

generate_queries = (
    prompt_perspectives 
    | ChatOpenAI(temperature=0) 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [10]:
# This prints the LangChain Runnable (a chain object) itself

print(generate_queries)

first=ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an AI language model assistant. Your task is to generate five \ndifferent versions of the given user question to retrieve relevant documents from a vector \ndatabase. By generating multiple perspectives on the user question, your goal is to help\nthe user overcome some of the limitations of the distance-based similarity search. \nProvide these alternative questions separated by newlines. Original question: {question}'), additional_kwargs={})]) middle=[ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False

In [11]:
# To see the actual multiple versions made by the LLM, we do this

question = "What is task decomposition for LLM agents?"
multi_queries = generate_queries.invoke({"question": question})

print(multi_queries)

['1. How do LLM agents utilize task decomposition in their operations?', '2. Can you explain the concept of task decomposition as applied to LLM agents?', '3. In what ways do LLM agents benefit from task decomposition strategies?', '4. What role does task decomposition play in the functioning of LLM agents?', '5. How is task decomposition implemented by LLM agents to enhance their performance?']


### Retrieval Chain

In [12]:
from langchain_core.load import dumps, loads

In [14]:
def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string for deduplication
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return back to Document objects
    return [loads(doc) for doc in unique_docs]

# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})

print("The retrieved docs are : \n", docs)
print("\nNumber of docs retrieved are : ", len(docs))

The retrieved docs are : 
 [Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Finite context length: The restricted context capacity limits the inclusion of historical information, detailed instructions, API call context, and responses. The design of the system has to work with this limited communication bandwidth, while mechanisms like self-reflection to learn from past mistakes would benefit a lot from long or infinite context windows. Although vector stores and retrieval can provide access to a larger knowledge pool, their representation power is not as powerful as full attention.\n\n\nChallenges in long-term planning and task decomposition: Planning over a lengthy history and effectively exploring the solution space remain challenging. LLMs struggle to adjust plans when faced with unexpected errors, making them less robust compared to humans who learn from trial and error.\n\n\nReliability of natural language interface: Current agen

### Final Generation

In [15]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

In [16]:
# Standard RAG Answer Prompt. So it seems in the final chain, we have to give all the retrieved text and the original
# question to the LLM

template = """
Answer the following question based on this context: {context} 
Question: {question}
""" 

prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(temperature=0) 

# Final Chain
final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
) 

# Test 
question = "What is task decomposition for LLM agents?"
response = final_rag_chain.invoke({"question":question})

print(response)

Task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals. This enables the agent to efficiently handle complex tasks by transforming them into multiple manageable tasks, shedding light on the interpretation of the model's thinking process. Task decomposition can be achieved through techniques such as Chain of Thought (CoT) and Tree of Thoughts, as well as through simple prompting, task-specific instructions, or human inputs.


## 2) RAG Fusion 

![RAG-Fusion Image](Images/RAG-Fusion.png)

### RAG Fusion generates multiple query variations, retrieves documents for each, and combines results using reciprocal rank fusion to improve retrieval relevance

In [17]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.load import dumps, loads
from langchain_core.runnables import RunnablePassthrough

In [18]:
llm = ChatOpenAI(temperature = 0)

### RAG-Fusion: Query Generation

In [19]:
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [20]:
# Generates the queries and splits them into a Python list

generate_queries = (
    prompt_rag_fusion 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [24]:
# To see the actual multiple queries returned by the LLM 

LLM_Multiple = generate_queries.invoke({'question' : question})
print(LLM_Multiple)
print("\nNumber or questions given: ", len(LLM_Multiple))

['1. How do LLM agents use task decomposition in their decision-making process?', '2. Benefits of task decomposition for LLM agents in problem-solving tasks', '3. Examples of task decomposition strategies used by LLM agents', '4. Challenges and limitations of task decomposition for LLM agents in complex environments']

Number or questions given:  4


### Reciprocal RAG Fusion (RRF) Algorithm

In [25]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            
            doc_str = dumps(doc)
        
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    return [doc for doc, score in reranked_results]

In [30]:
# Helper function to extract text from Document objects

def format_docs(docs):
    """ Extracts just the page content from the Document objects """
    return "\n\n".join(doc.page_content for doc in docs)

### Assemble Retrieval Chain

In [27]:
# Generate queries -> Map retriever -> Rank via RRF -> Format to clean text

retrieval_chain_rag_fusion = (
    generate_queries 
    | retriever.map() 
    | reciprocal_rank_fusion 
    | format_docs
)

### Final RAG Chain

In [28]:
template = """Answer the following question based on this context: {context} 
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

In [29]:
# Test the pipeline

question = "What is task decomposition for LLM agents?"
response = final_rag_chain.invoke({"question": question})

print(response)

Task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals to enable efficient handling of complex tasks. This process allows the agent to think step by step and decompose hard tasks into smaller and simpler steps, ultimately transforming big tasks into multiple manageable tasks. Task decomposition can be done through techniques like Chain of Thought (CoT) and Tree of Thoughts, which explore multiple reasoning possibilities at each step and generate multiple thoughts per step to create a tree structure. Additionally, task decomposition can be facilitated through simple prompting, task-specific instructions, or human inputs.


## 3) Decomposition

### Decomposition breaks a complex question into multiple sub-questions, retrieves answers for each, and then synthesizes them to answer the original question

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from langchain_classic import hub

In [32]:
# Initialize LLM

llm = ChatOpenAI(temperature=0)

In [33]:
# Document formatter 

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Decomposition Generation

In [34]:
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answered in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template) 

generate_queries_decomposition = ( 
    prompt_decomposition 
    | llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [36]:
# Let's test and see how the LLM makes decomposed questions 

question = "What are the main components of an LLM-powered autonomous agent system?"

sub_questions = generate_queries_decomposition.invoke({"question" : question})
print(sub_questions)
print("\nThe number of sub questions made are: ", len(sub_questions))

['1. What is LLM technology and how does it work in autonomous agent systems?', '2. What are the specific components that make up an LLM-powered autonomous agent system?', '3. How do the main components of an LLM-powered autonomous agent system interact with each other to enable autonomous behavior?']

The number of sub questions made are:  3


### Answer Recursively (Multi-hop Reasoning) (Chain-of-Thought Retrieval) 

![Decomposition_Recursively](Images/Decomposition_Recursively.png)

In [38]:
template_recursive = """Here is the question you need to answer:
\n --- \n {question} \n --- \n
Here is any available background question + answer pairs:
\n --- \n {q_a_pairs} \n --- \n
Here is additional context relevant to the question: 
\n --- \n {context} \n --- \n
Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template_recursive)

In [39]:
def format_qa_pair(q, a):
    return f"Question: {q}\nAnswer: {a}\n\n".strip() 

q_a_pairs = ""

for q in sub_questions:
    rag_chain = (
        {"context": itemgetter("question") | retriever | format_docs, 
         "question": itemgetter("question"),
         "q_a_pairs": itemgetter("q_a_pairs")} 
        | decomposition_prompt
        | llm
        | StrOutputParser()
    )

    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair

print("Recursive Result:\n", q_a_pairs, "\n" + "="*50)

Recursive Result:
 
---
Question: 1. What is LLM technology and how does it work in autonomous agent systems?
Answer: LLM technology, which stands for Large Language Model, is utilized as the core controller in autonomous agent systems. In these systems, LLM functions as the agent's brain and is complemented by key components such as planning, memory, and tool use. 

In terms of planning, the agent can break down large tasks into smaller subgoals for efficient handling of complex tasks. It can also engage in self-reflection and refinement, learning from past actions and mistakes to improve future results.

Memory in LLM-powered autonomous agent systems can involve utilizing an external classical planner for long-horizon planning. This approach involves translating the planning problem into a Planning Domain Definition Language (PDDL), requesting a classical planner to generate a plan, and then translating the plan back into natural language.

Additionally, LLM-powered agents can use to

### Answer Individually (Sub-query Generation)

![Decomposition_Individually](Images/Decomposition_Individually.png)

In [ ]:
prompt_rag = hub.pull("rlm/rag-prompt")

def retrieve_and_rag(main_question, sub_question_generator_chain):
    """RAG on each sub-question independently"""
    
    sub_questions = sub_question_generator_chain.invoke({"question": main_question})
    rag_results = []
    
    for sub_question in sub_questions:
        retrieved_docs = retriever.invoke(sub_question)
        
        answer = (
            prompt
            | llm
            | StrOutputParser()
        ).invoke({
            "context": format_docs(retrieved_docs),
            "question": sub_question
        })
        
        rag_results.append(answer)
    
    return rag_results, sub_questions

answers, questions_list = retrieve_and_rag(question, generate_queries_decomposition)

In [49]:
print("Answers to each sub question: \n", answers)
print("\nNumber of answers: ", len(answers)) 

print("\nSub questions: \n", questions_list)
print("\nNumber of sub questions made by LLM: ", len(questions_list)) 

Answers to each sub question: 
 ["LLM technology refers to large language models that serve as the core controller in autonomous agent systems. In these systems, LLM functions as the agent's brain, complemented by components such as planning, memory, self-reflection, and tool use. LLM is capable of generating well-written copies, stories, essays, and programs, and can also serve as a powerful general problem solver. The technology allows the agent to break down tasks into smaller subgoals, reflect on past actions, learn from mistakes, and interact with the environment through a combination of task-specific actions and natural language reasoning.", 'The specific components that make up an LLM-powered autonomous agent system include planning (subgoal and decomposition, reflection and refinement), memory (short-term and long-term memory), and tool use (calling external APIs for additional information).', 'The main components of an LLM-powered autonomous agent system interact with each oth

In [50]:
def format_qa_pairs(q_list, a_list):
    """Format multiple Q and A pairs using zip and enumerate"""
    
    formatted_string = ""
    for i, (q, a) in enumerate(zip(q_list, a_list), start=1):
        formatted_string += f"Question {i}: {q}\nAnswer {i}: {a}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions_list, answers)

In [51]:
print(context)

Question 1: 1. What is LLM technology and how does it work in autonomous agent systems?
Answer 1: LLM technology refers to large language models that serve as the core controller in autonomous agent systems. In these systems, LLM functions as the agent's brain, complemented by components such as planning, memory, self-reflection, and tool use. LLM is capable of generating well-written copies, stories, essays, and programs, and can also serve as a powerful general problem solver. The technology allows the agent to break down tasks into smaller subgoals, reflect on past actions, learn from mistakes, and interact with the environment through a combination of task-specific actions and natural language reasoning.

Question 2: 2. What are the specific components that make up an LLM-powered autonomous agent system?
Answer 2: The specific components that make up an LLM-powered autonomous agent system include planning (subgoal and decomposition, reflection and refinement), memory (short-term an

In [52]:
# Final Synthesis Prompt

template_synthesis = """Here is a set of Q+A pairs: {context}. 
Use these to synthesize an answer to the following overarching question: {question}
"""
prompt_synthesis = ChatPromptTemplate.from_template(template_synthesis)

final_rag_chain = (
    prompt_synthesis
    | llm
    | StrOutputParser()
)

final_synthesis_answer = final_rag_chain.invoke({"context": context, "question": question})

print("Synthesized Result:\n", final_synthesis_answer)

Synthesized Result:
 The main components of an LLM-powered autonomous agent system include planning, memory, and tool use. Planning involves breaking down tasks into subgoals and reflecting on past actions to improve decision-making. Memory includes short-term memory for in-context learning and long-term memory for retaining information. Tool use allows the agent to call external APIs for additional information, enhancing decision-making capabilities. These components work together to enable autonomous behavior in the agent system.


## Step-Back

![StepBack Image](Images/StepBack.png)

### Step-Back refers to paraphrasing a specific question into a broader, more generic version to improve retrieval and answer quality in RAG systems

In [53]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [54]:
# Helper function to extract text from Document objects

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Few-Shot Examples

In [55]:
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "What can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "What is Jan Sindel’s personal history?",
    },
]

In [56]:
# Transform these into example messages for the Chat Model

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

### Step-Back Question Generator

In [58]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:",
        ),
        
        # Inject the few-shot examples
        few_shot_prompt,
        
        # The new question to process
        ("user", "{question}"),
    ]
)

print("This is how the prompt looks like: \n\n", prompt)

This is how the prompt looks like: 

 input_variables=['question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': 'Could the members of The Police perform lawful arrests?', 'output': 'What can the members of The Police do?'}, {'input': 'Jan Sindel’s was born in what country?', 'output': 'What is Jan Sindel’s personal history?'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, t

In [59]:
llm = ChatOpenAI(temperature=0)

generate_queries_step_back = prompt | llm | StrOutputParser()

In [62]:
# Let's test this and see what the generated Step-Back query looks like 

question = "What is task decomposition for LLM agents?"

step_back_query = generate_queries_step_back.invoke({"question": question})
print("The Step-Back query is: \n", step_back_query)

The Step-Back query is: 
 How do LLM agents break down tasks into smaller steps?


### Final Answer Generation Prompt

In [61]:
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. 
Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant. 

Normal Context: {normal_context}. 

Step-Back Context: {step_back_context}. 

Original Question: {question}. 

Answer: 

"""

response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

### Assemble the Full RAG Chain

In [63]:
chain = (
    {
        # Route 1: Retrieve context using the original, highly-specific question
        "normal_context": itemgetter("question") | retriever | format_docs,
        
        # Route 2: Generate a broad question, then retrieve context using that broad question
        "step_back_context": generate_queries_step_back | retriever | format_docs,
        
        # Route 3: Pass the original question straight through
        "question": itemgetter("question"),
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

### Execution 

In [65]:
question = "What is task decomposition for LLM agents?"

# Execute the final chain
response = chain.invoke({"question": question})
print(response)

Task decomposition for LLM agents refers to the process of breaking down large and complex tasks into smaller, more manageable subgoals. This approach allows LLM-powered agents to efficiently handle intricate tasks by dividing them into simpler steps that can be tackled sequentially. Task decomposition is essential for enhancing the performance of LLM agents on complex tasks, as it enables them to utilize more test-time computation and shed light on the interpretation of the model's thinking process.

One common technique for task decomposition is the Chain of Thought (CoT), which prompts the model to "think step by step" and decompose hard tasks into smaller and simpler steps. This technique transforms big tasks into multiple manageable tasks and helps in understanding the model's thinking process. Additionally, the Tree of Thoughts extends CoT by exploring multiple reasoning possibilities at each step, creating a tree structure of multiple thought steps and generating multiple though

## Hypothetical Document Embedding (HyDE)

![HyDE Image](Images/HyDE.png)

### HyDE generates a hypothetical answer to a question using an LLM, embeds it, and uses this embedding to retrieve relevant documents from a vector store

In [66]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [67]:
# Extracts the text from the retrieved Document objects

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### HyDE Document Generation

In [68]:
template_hyde = """Please write a scientific paper passage to answer the question, 
Question: {question}. 

Passage: 
"""

prompt_hyde = ChatPromptTemplate.from_template(template_hyde)

In [73]:
llm = ChatOpenAI(temperature=0)

# Chain that generates the hypothetical document
generate_docs_for_retrieval = (
    prompt_hyde | llm | StrOutputParser() 
)

### Retrieval Pipeline

In [74]:
# This chain generates the fake doc, uses it to search the vector store, and formats the results

retrieval_chain = generate_docs_for_retrieval | retriever | format_docs

### Final RAG Answer Generation

In [75]:
template_rag = """Answer the following question based on this context: {context}. 
Question: {question}.
"""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

In [77]:
# Let's see how the LLM generates a fake hypothetical answer 

question = "What is task decomposition for LLM agents?"

hypothetical_answer = generate_docs_for_retrieval.invoke({"question": question})
print("The hypothetical question is: \n", hypothetical_answer)

The hypothetical question is: 
 Task decomposition is a fundamental concept in the field of reinforcement learning and artificial intelligence, particularly for Large Language Models (LLMs) agents. Task decomposition refers to the process of breaking down a complex task into smaller, more manageable sub-tasks that can be solved independently or sequentially. This approach allows LLM agents to effectively tackle complex problems by dividing them into smaller, more manageable components.

In the context of LLM agents, task decomposition is crucial for improving the efficiency and effectiveness of the learning process. By breaking down a complex task into smaller sub-tasks, LLM agents can focus on solving each sub-task individually, which can lead to faster learning and better performance overall. Additionally, task decomposition can help LLM agents to generalize their knowledge and skills across different tasks, as they can apply the same decomposition strategy to new problems.

Overall,

### Assemble the Master Chain

In [78]:
final_rag_chain = (
    {
        # Generate fake doc -> Retrieve real docs -> Format to text
        "context": retrieval_chain, 
        
        # Pass the original question through
        "question": itemgetter("question")
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

### Execution

In [79]:
# Execute the full pipeline

question = "What is task decomposition for LLM agents?"

response = final_rag_chain.invoke({"question": question})
print(response)

Task decomposition for LLM agents involves breaking down complex tasks into smaller and simpler steps, allowing the agent to think step by step and utilize more test-time computation. This process transforms big tasks into multiple manageable tasks, shedding light on the model's thinking process. Task decomposition can be achieved through techniques like Chain of Thought (CoT) and Tree of Thoughts, as well as through task-specific instructions or human inputs. Additionally, some approaches, like LLM+P, involve outsourcing the planning step to an external classical planner using the Planning Domain Definition Language (PDDL) as an intermediate interface.
